In [1]:
print("Hello, World!")

Hello, World!


# 02_4 DQA-CWA Paper Protocol Evaluation

This notebook is the dedicated paper-protocol evaluation pass. It runs the shared per-weather validation script against the selected DQA workspace, writes the summary artifacts under `validation_reports/`, and then reshapes the results so it is easy to see whether the paper-style splits tell a different story from the training-time `server_cloudy_val` view.

In [2]:
from __future__ import annotations

import json
import re
import shlex
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image as NotebookImage, display

try:
    import seaborn as sns
except ModuleNotFoundError:
    sns = None


def find_repo_root(start: Optional[Path] = None) -> Path:
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    required = (
        "dynamic_quality_aware_classwise_aggregation/run_dqa_cwa_fedsto.py",
        "navigating_data_heterogeneity/setup_fedsto_exact_reproduction.py",
    )
    candidate_dirs = []
    for base in (start, *start.parents):
        candidate_dirs.extend(
            [
                base,
                base / "Object_Detection",
                base / "masters_research" / "Object_Detection",
            ]
        )
    for candidate in candidate_dirs:
        if all((candidate / marker).exists() for marker in required):
            return candidate.resolve()
    raise FileNotFoundError("Could not locate the Object_Detection repository root.")


REPO_ROOT = find_repo_root()
DQA_ROOT = REPO_ROOT / "dynamic_quality_aware_classwise_aggregation"
NAV_ROOT = REPO_ROOT / "navigating_data_heterogeneity"
EVAL_SCRIPT = DQA_ROOT / "evaluate_paper_protocol.py"
NOTEBOOK_GENERATOR = DQA_ROOT / "generate_dqa_cwa_notebook.py"

WORKSPACE_NAME = "efficientteacher_dqa_cwa_corrected_12h"
ALTERNATIVE_WORKSPACES = [
    "efficientteacher_dqa_cwa_corrected_12h",
    "efficientteacher_dqa_cwa_14h",
    "efficientteacher_dqa_cwa_exact",
    "efficientteacher_dqa_cwa",
]
WORK_ROOT = DQA_ROOT / WORKSPACE_NAME
VALIDATION_ROOT = WORK_ROOT / "validation_reports"
RUN_ROOT = VALIDATION_ROOT / "paper_protocol_val_runs"
LOG_ROOT = VALIDATION_ROOT / "paper_protocol_logs"
FEDSTO_WORK_ROOT = NAV_ROOT / "efficientteacher_fedsto"
FEDSTO_PAPER_EVAL_SUMMARY = FEDSTO_WORK_ROOT / "validation_reports" / "paper_protocol_eval_summary.csv"

preferred_python = Path("/root/micromamba/envs/al_yolov8/bin/python")
PYTHON_BIN = preferred_python if preferred_python.exists() else Path(sys.executable)

if sns is not None:
    sns.set_theme(style="whitegrid", context="talk")
else:
    plt.style.use("ggplot")
pd.options.display.max_columns = 200
pd.options.display.max_rows = 200


def modified_utc(path: Path) -> str:
    if not path.exists():
        return ""
    return datetime.fromtimestamp(path.stat().st_mtime, tz=timezone.utc).isoformat()


def parse_float(value) -> float | None:
    if value in (None, "") or pd.isna(value):
        return None
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def best_phase2_round_from_summary(summary_csv: Path, basis: str) -> int | None:
    if not summary_csv.exists():
        return None
    metric_col = {
        "map50": "final_metrics/mAP_0.5",
        "map50_95": "final_metrics/mAP_0.5:0.95",
    }[basis]
    summary = pd.read_csv(summary_csv)
    phase2 = summary[(summary["phase"] == 2) & (summary["role"] == "server")].copy()
    if phase2.empty or metric_col not in phase2.columns:
        return None
    phase2 = phase2.sort_values(metric_col, ascending=False)
    return int(phase2.iloc[0]["round"])


def auto_checkpoint_specs(workspace: Path, best_basis: str) -> list[tuple[str, Path]]:
    history_path = workspace / "history.json"
    history = json.loads(history_path.read_text(encoding="utf-8")) if history_path.exists() else []
    global_dir = workspace / "global_checkpoints"
    specs: list[tuple[str, Path]] = []
    seen: set[Path] = set()

    def add(label: str, path: Path) -> None:
        resolved = path.resolve()
        if not path.exists() or resolved in seen:
            return
        seen.add(resolved)
        specs.append((label, resolved))

    add("warmup_global", global_dir / "round000_warmup.pt")

    phase1 = [entry for entry in history if int(entry.get("phase", 0)) == 1]
    if phase1:
        add(f"phase1_round{int(phase1[-1]['round']):03d}_global", Path(phase1[-1]["global"]))

    summary_csv = workspace / "validation_reports" / "tables" / "training_run_summary.csv"
    best_round = best_phase2_round_from_summary(summary_csv, best_basis)
    if best_round is not None:
        add(
            f"phase2_best_server_cloudy_round{best_round:03d}_global",
            global_dir / f"phase2_round{best_round:03d}_global.pt",
        )

    phase2 = [entry for entry in history if int(entry.get("phase", 0)) == 2]
    if phase2:
        add(f"phase2_round{int(phase2[-1]['round']):03d}_global", Path(phase2[-1]["global"]))
    return specs


def plot_bar(ax, data: pd.DataFrame, *, x: str, y: str, hue: str | None = None):
    if sns is not None:
        return sns.barplot(data=data, x=x, y=y, hue=hue, ax=ax)

    if hue is not None and hue in data.columns:
        pivot = data.pivot_table(index=x, columns=hue, values=y, aggfunc="mean")
        pivot.plot(kind="bar", ax=ax)
    else:
        grouped = data.groupby(x, as_index=False)[y].mean()
        ax.bar(grouped[x], grouped[y])
    return ax


def plot_heatmap(
    ax,
    matrix,
    *,
    row_labels: list[str],
    col_labels: list[str],
    title: str,
    cmap: str = "viridis",
    fmt: str = ".3f",
    annotate: bool = True,
):
    values = np.asarray(matrix, dtype=float)
    if sns is not None:
        sns.heatmap(
            values,
            ax=ax,
            cmap=cmap,
            annot=annotate,
            fmt=fmt,
            xticklabels=col_labels,
            yticklabels=row_labels,
        )
    else:
        image = ax.imshow(values, aspect="auto", cmap=cmap)
        ax.figure.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
        ax.set_xticks(range(len(col_labels)))
        ax.set_xticklabels(col_labels, rotation=35, ha="right")
        ax.set_yticks(range(len(row_labels)))
        ax.set_yticklabels(row_labels)
    ax.set_title(title)
    ax.set_xlabel("split")
    ax.set_ylabel("checkpoint")
    return ax


print("repo_root:", REPO_ROOT)
print("workspace_name:", WORKSPACE_NAME)
print("workspace:", WORK_ROOT)
print("validation_root:", VALIDATION_ROOT)
print("python:", PYTHON_BIN)
print("alternatives:", ", ".join(ALTERNATIVE_WORKSPACES))

repo_root: /app/Object_Detection
workspace_name: efficientteacher_dqa_cwa_corrected_12h
workspace: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa_cwa_corrected_12h
validation_root: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa_cwa_corrected_12h/validation_reports
python: /root/micromamba/envs/al_yolov8/bin/python
alternatives: efficientteacher_dqa_cwa_corrected_12h, efficientteacher_dqa_cwa_14h, efficientteacher_dqa_cwa_exact, efficientteacher_dqa_cwa


## 1. Workspace Status and Auto-Selected Checkpoints

This notebook is focused on the paper-style validation protocol. It first checks whether the selected workspace exists and which checkpoints the shared evaluation script would pick by default.

In [3]:
manifest_path = WORK_ROOT / "manifest.json"
history_path = WORK_ROOT / "history.json"
summary_csv = VALIDATION_ROOT / "tables" / "training_run_summary.csv"
paper_summary_csv = VALIDATION_ROOT / "paper_protocol_eval_summary.csv"
paper_manifest_json = VALIDATION_ROOT / "paper_protocol_eval_manifest.json"

history = json.loads(history_path.read_text(encoding="utf-8")) if history_path.exists() else []
manifest = json.loads(manifest_path.read_text(encoding="utf-8")) if manifest_path.exists() else {}
auto_specs_preview = auto_checkpoint_specs(WORK_ROOT, "map50")

artifact_rows = [
    {"artifact": "workspace", "path": str(WORK_ROOT), "exists": WORK_ROOT.exists(), "modified_utc": modified_utc(WORK_ROOT)},
    {"artifact": "manifest", "path": str(manifest_path), "exists": manifest_path.exists(), "modified_utc": modified_utc(manifest_path)},
    {"artifact": "history", "path": str(history_path), "exists": history_path.exists(), "modified_utc": modified_utc(history_path)},
    {"artifact": "training_summary_csv", "path": str(summary_csv), "exists": summary_csv.exists(), "modified_utc": modified_utc(summary_csv)},
    {"artifact": "paper_eval_summary_csv", "path": str(paper_summary_csv), "exists": paper_summary_csv.exists(), "modified_utc": modified_utc(paper_summary_csv)},
    {"artifact": "paper_eval_manifest_json", "path": str(paper_manifest_json), "exists": paper_manifest_json.exists(), "modified_utc": modified_utc(paper_manifest_json)},
]
display(pd.DataFrame(artifact_rows))

history_rows = {
    "completed_phase1_rounds": sum(1 for entry in history if int(entry.get("phase", 0)) == 1),
    "completed_phase2_rounds": sum(1 for entry in history if int(entry.get("phase", 0)) == 2),
    "history_entries": len(history),
    "classes": manifest.get("classes", []),
    "client_weathers": [client.get("weather") for client in manifest.get("clients", [])],
}
history_rows

if auto_specs_preview:
    display(
        pd.DataFrame(
            [{"checkpoint_label": label, "checkpoint_path": str(path)} for label, path in auto_specs_preview]
        )
    )
else:
    print("No auto-selected checkpoints are available yet for this workspace.")

,artifact,path,exists,modified_utc
0,workspace,/app/Object_Detection/dynamic_quality_aware_cl...,True,2026-04-27T02:40:47.966515+00:00
1,manifest,/app/Object_Detection/dynamic_quality_aware_cl...,True,2026-04-26T16:54:17.226219+00:00
2,history,/app/Object_Detection/dynamic_quality_aware_cl...,True,2026-04-27T01:08:10.211053+00:00
3,training_summary_csv,/app/Object_Detection/dynamic_quality_aware_cl...,True,2026-04-27T02:41:25.082344+00:00
4,paper_eval_summary_csv,/app/Object_Detection/dynamic_quality_aware_cl...,False,
5,paper_eval_manifest_json,/app/Object_Detection/dynamic_quality_aware_cl...,False,


,checkpoint_label,checkpoint_path
0,warmup_global,/app/Object_Detection/dynamic_quality_aware_cl...
1,phase1_round014_global,/app/Object_Detection/dynamic_quality_aware_cl...
2,phase2_best_server_cloudy_round024_global,/app/Object_Detection/dynamic_quality_aware_cl...
3,phase2_round027_global,/app/Object_Detection/dynamic_quality_aware_cl...


## 2. Paper-Eval Controls

The defaults below run the full paper-style split set on the current workspace. Set `RUN_PAPER_EVAL = False` if you only want to inspect already-generated outputs.

In [4]:
RUN_PAPER_EVAL = True
DRY_RUN = False
BEST_BASIS = "map50"
EVAL_SPLITS = "cloudy,overcast,rainy,snowy,total"
BATCH_SIZE = 8
IMGSZ = 640
CONF_THRES = 0.001
IOU_THRES = 0.6
DEVICE = ""
PLOTS = True
VERBOSE = False

# Leave this empty to use the script's auto checkpoint selection:
#   warmup_global, phase1 final, phase2 best-by-BEST_BASIS, phase2 final.
# Otherwise add items like:
# EXPLICIT_CHECKPOINTS = [
#     ("my_label", WORK_ROOT / "global_checkpoints" / "phase2_round024_global.pt"),
# ]
EXPLICIT_CHECKPOINTS: list[tuple[str, Path]] = []

{
    "run_paper_eval": RUN_PAPER_EVAL,
    "dry_run": DRY_RUN,
    "best_basis": BEST_BASIS,
    "eval_splits": EVAL_SPLITS,
    "batch_size": BATCH_SIZE,
    "imgsz": IMGSZ,
    "conf_thres": CONF_THRES,
    "iou_thres": IOU_THRES,
    "device": DEVICE,
    "plots": PLOTS,
    "verbose": VERBOSE,
    "explicit_checkpoints": [(label, str(path)) for label, path in EXPLICIT_CHECKPOINTS],
}

{'run_paper_eval': True,
 'dry_run': False,
 'best_basis': 'map50',
 'eval_splits': 'cloudy,overcast,rainy,snowy,total',
 'batch_size': 8,
 'imgsz': 640,
 'conf_thres': 0.001,
 'iou_thres': 0.6,
 'device': '',
 'plots': True,
 'verbose': False,
 'explicit_checkpoints': []}

## 3. Run the Shared Paper Protocol Evaluation

This calls the shared `evaluate_paper_protocol.py` entrypoint. It writes the CSV, Markdown summary, manifest, logs, and per-split validation run directories under `validation_reports/`.

In [ ]:
checkpoint_specs = EXPLICIT_CHECKPOINTS if EXPLICIT_CHECKPOINTS else auto_checkpoint_specs(WORK_ROOT, BEST_BASIS)

eval_cmd = [
    str(PYTHON_BIN),
    str(EVAL_SCRIPT),
    "--workspace",
    str(WORK_ROOT),
    "--splits",
    EVAL_SPLITS,
    "--best-basis",
    BEST_BASIS,
    "--batch-size",
    str(BATCH_SIZE),
    "--imgsz",
    str(IMGSZ),
    "--conf-thres",
    str(CONF_THRES),
    "--iou-thres",
    str(IOU_THRES),
]
if DEVICE:
    eval_cmd.extend(["--device", DEVICE])
if PLOTS:
    eval_cmd.append("--plots")
else:
    eval_cmd.append("--no-plots")
if VERBOSE:
    eval_cmd.append("--verbose")
if DRY_RUN:
    eval_cmd.append("--dry-run")
for label, path in checkpoint_specs:
    eval_cmd.extend(["--checkpoint", f"{label}={path}"])

print("Resolved checkpoints:")
if checkpoint_specs:
    display(pd.DataFrame([{"checkpoint_label": label, "checkpoint_path": str(path)} for label, path in checkpoint_specs]))
else:
    print("No checkpoints resolved.")

print("Command:")
print(" ".join(shlex.quote(part) for part in eval_cmd))

if RUN_PAPER_EVAL:
    if not WORK_ROOT.exists():
        raise FileNotFoundError(f"Workspace does not exist: {WORK_ROOT}")
    subprocess.run(eval_cmd, cwd=REPO_ROOT, check=True)
else:
    print("Set RUN_PAPER_EVAL = True and rerun this cell to execute the paper protocol.")

if paper_summary_csv.exists():
    display(pd.read_csv(paper_summary_csv))
else:
    print("No paper summary CSV found yet:", paper_summary_csv)

Resolved checkpoints:


,checkpoint_label,checkpoint_path
0,warmup_global,/app/Object_Detection/dynamic_quality_aware_cl...
1,phase1_round014_global,/app/Object_Detection/dynamic_quality_aware_cl...
2,phase2_best_server_cloudy_round024_global,/app/Object_Detection/dynamic_quality_aware_cl...
3,phase2_round027_global,/app/Object_Detection/dynamic_quality_aware_cl...


Command:
/root/micromamba/envs/al_yolov8/bin/python /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/evaluate_paper_protocol.py --workspace /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa_cwa_corrected_12h --splits cloudy,overcast,rainy,snowy,total --best-basis map50 --batch-size 8 --imgsz 640 --conf-thres 0.001 --iou-thres 0.6 --plots --checkpoint warmup_global=/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa_cwa_corrected_12h/global_checkpoints/round000_warmup.pt --checkpoint phase1_round014_global=/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa_cwa_corrected_12h/global_checkpoints/phase1_round014_global.pt --checkpoint phase2_best_server_cloudy_round024_global=/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa_cwa_corrected_12h/global_checkpoints/phase2_round024_global.pt --checkpoint phase2_round027_global=/app/Obje

## 4. Paper Protocol Results

This section reshapes the output into the two views that usually answer the question fastest: per-checkpoint totals and checkpoint-by-split heatmaps.

In [ ]:
if not paper_summary_csv.exists():
    print("No paper summary CSV found yet:", paper_summary_csv)
else:
    paper_eval_df = pd.read_csv(paper_summary_csv)
    display(paper_eval_df)

    ok_rows = paper_eval_df[paper_eval_df["status"] == "ok"].copy()
    if ok_rows.empty:
        print("Paper summary exists, but no successful rows were recorded yet.")
    else:
        total_rows = (
            ok_rows[ok_rows["split"] == "total"]
            .sort_values("map50", ascending=False)
            .reset_index(drop=True)
        )
        if not total_rows.empty:
            print("Total-split ranking by mAP@0.5")
            display(total_rows[["checkpoint_label", "precision", "recall", "map50", "map50_95"]].round(4))

        weather_rows = ok_rows[ok_rows["split"] != "total"].copy()
        if not weather_rows.empty:
            mean_weather = (
                weather_rows.groupby("checkpoint_label", as_index=False)[["precision", "recall", "map50", "map50_95"]]
                .mean()
                .sort_values("map50", ascending=False)
            )
            print("Mean across paper weather splits")
            display(mean_weather.round(4))

        fig, axes = plt.subplots(1, 2, figsize=(18, 5))
        if not total_rows.empty:
            plot_bar(axes[0], total_rows, x="checkpoint_label", y="map50")
            plot_bar(axes[1], total_rows, x="checkpoint_label", y="map50_95")
            axes[0].set_title("total split mAP@0.5")
            axes[1].set_title("total split mAP@0.5:0.95")
            for ax in axes:
                ax.tick_params(axis="x", rotation=20)
                ax.set_xlabel("")
        else:
            for ax in axes:
                ax.set_visible(False)
        plt.tight_layout()
        plt.show()

        pivot50 = ok_rows.pivot_table(index="checkpoint_label", columns="split", values="map50", aggfunc="mean")
        pivot5095 = ok_rows.pivot_table(index="checkpoint_label", columns="split", values="map50_95", aggfunc="mean")

        if not pivot50.empty:
            fig, axes = plt.subplots(1, 2, figsize=(18, max(4, 1.4 * len(pivot50))))
            plot_heatmap(
                axes[0],
                pivot50.fillna(np.nan).values,
                row_labels=list(pivot50.index),
                col_labels=list(pivot50.columns),
                title="mAP@0.5 by checkpoint and split",
            )
            plot_heatmap(
                axes[1],
                pivot5095.fillna(np.nan).values,
                row_labels=list(pivot5095.index),
                col_labels=list(pivot5095.columns),
                title="mAP@0.5:0.95 by checkpoint and split",
            )
            plt.tight_layout()
            plt.show()

## 5. DQA vs FedSTO Paper-Summary Snapshot

If the baseline FedSTO paper summary already exists, line up the total split rows so we can quickly see whether the paper protocol changes the story.

In [ ]:
frames = []
if paper_summary_csv.exists():
    dqa_df = pd.read_csv(paper_summary_csv)
    dqa_df.insert(0, "method", "DQA-CWA")
    frames.append(dqa_df)
if FEDSTO_PAPER_EVAL_SUMMARY.exists():
    fedsto_df = pd.read_csv(FEDSTO_PAPER_EVAL_SUMMARY)
    fedsto_df.insert(0, "method", "FedSTO")
    frames.append(fedsto_df)

if not frames:
    print("No paper-protocol summary CSVs found yet.")
else:
    compare_df = pd.concat(frames, ignore_index=True)
    display(compare_df.round(4))

    ok_total = compare_df[(compare_df["status"] == "ok") & (compare_df["split"] == "total")].copy()
    if not ok_total.empty:
        best_total = (
            ok_total.sort_values(["method", "map50"], ascending=[True, False])
            .groupby("method", as_index=False)
            .head(1)
            .reset_index(drop=True)
        )
        print("Best total split by method")
        display(best_total[["method", "checkpoint_label", "precision", "recall", "map50", "map50_95"]].round(4))

## 6. Visual Artifact Preview

When plots are enabled, the underlying validation runs save PR curves and confusion matrices. This cell previews the newest run that has both files.

In [ ]:
preview_dir = next(
    (
        path
        for path in sorted(RUN_ROOT.glob("*"), key=lambda candidate: candidate.stat().st_mtime, reverse=True)
        if path.is_dir() and (path / "PR_curve.png").exists() and (path / "confusion_matrix.png").exists()
    ),
    None,
)

if preview_dir is None:
    print("No completed paper-eval artifact directory with PR/confusion plots was found under:", RUN_ROOT)
else:
    print("Preview directory:", preview_dir)
    display(NotebookImage(filename=str(preview_dir / "PR_curve.png"), width=700))
    display(NotebookImage(filename=str(preview_dir / "confusion_matrix.png"), width=700))

## 7. Artifact Index

This is the short click-list for the generated summary CSV, manifest JSON, logs, and validation run directories.

In [ ]:
def artifact_row(path: Path, label: str) -> dict:
    return {
        "label": label,
        "path": str(path),
        "exists": path.exists(),
        "modified_utc": modified_utc(path),
    }


artifact_rows = [
    artifact_row(EVAL_SCRIPT, "paper_eval_script"),
    artifact_row(NOTEBOOK_GENERATOR, "notebook_generator"),
    artifact_row(WORK_ROOT, "workspace"),
    artifact_row(manifest_path, "manifest"),
    artifact_row(history_path, "history"),
    artifact_row(summary_csv, "training_summary_csv"),
    artifact_row(paper_summary_csv, "paper_eval_summary_csv"),
    artifact_row(paper_manifest_json, "paper_eval_manifest_json"),
    artifact_row(VALIDATION_ROOT / "paper_protocol_eval_summary.md", "paper_eval_summary_md"),
    artifact_row(LOG_ROOT, "paper_eval_logs_dir"),
    artifact_row(RUN_ROOT, "paper_eval_run_dir"),
]
display(pd.DataFrame(artifact_rows))